# Configs — 03: Export / Import Catalog Config

**Persona:** Admin

**Goal:** Export all configs from a source (production) catalog, save them locally,
then import them into a target (staging) catalog. Verify that routing resolution
works correctly in the target after import.

**Use case:** Catalog cloning from production to staging — ensures staging mirrors
production driver and routing configuration without manual re-entry.

---

## Prerequisites

- DynaStore running and reachable at `DYNASTORE_BASE_URL`
- Both `SOURCE_CATALOG_ID` and `TARGET_CATALOG_ID` catalogs exist
- `DYNASTORE_ADMIN_TOKEN` is set with admin scope on both catalogs
- Source catalog already has at least one config set (run notebooks 01 or 02 first)

## Environment variables

| Variable | Default | Description |
|---|---|---|
| `DYNASTORE_BASE_URL` | `http://localhost:8080` | API base URL |
| `DYNASTORE_ADMIN_TOKEN` | _(empty)_ | Bearer token for admin operations |
| `SOURCE_CATALOG_ID` | `demo-catalog` | Source (production) catalog |
| `TARGET_CATALOG_ID` | `staging-catalog` | Target (staging) catalog |
| `COLLECTION_ID` | `sentinel2-l2a` | Collection to verify after import |

In [ ]:
import os
import json
import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL          = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
ADMIN_TOKEN       = os.environ.get("DYNASTORE_ADMIN_TOKEN", "")
SOURCE_CATALOG_ID = os.environ.get("SOURCE_CATALOG_ID", "demo-catalog")
TARGET_CATALOG_ID = os.environ.get("TARGET_CATALOG_ID", "staging-catalog")
COLLECTION_ID     = os.environ.get("COLLECTION_ID", "sentinel2-l2a")

headers = {"Authorization": f"Bearer {ADMIN_TOKEN}"} if ADMIN_TOKEN else {}
client  = httpx.Client(base_url=BASE_URL, headers=headers, timeout=60)

print(f"Base URL          : {BASE_URL}")
print(f"Source catalog    : {SOURCE_CATALOG_ID}")
print(f"Target catalog    : {TARGET_CATALOG_ID}")
print(f"Collection        : {COLLECTION_ID}")
print(f"Auth header       : {'set' if ADMIN_TOKEN else 'not set'}")

In [ ]:
# Step 1 — Export all configs from the source catalog
#
# The export includes all catalog-level configs (and optionally collection-level
# configs for all collections in the catalog). The export_timestamp in the
# response is informational only — it is NOT used for version locking on import.
resp = client.get(f"/admin/configs/{SOURCE_CATALOG_ID}/export")
assert resp.status_code == 200, (
    f"Expected 200, got {resp.status_code}: {resp.text[:400]}"
)

export_payload = resp.json()

# Count catalog-level configs
catalog_configs = export_payload.get("catalog_configs", {})
collection_configs = export_payload.get("collection_configs", {})
export_timestamp   = export_payload.get("export_timestamp", "not reported")

print(f"Export from {SOURCE_CATALOG_ID!r}:")
print(f"  export_timestamp : {export_timestamp}")
print(f"  catalog_configs  : {len(catalog_configs)} keys")
print(f"  collection_configs: {len(collection_configs)} collections")
print()
print("Catalog config keys:")
for k in catalog_configs:
    print(f"  {k}")

In [ ]:
# Step 2 — Save export to a local file
export_path = "export.json"
with open(export_path, "w") as fh:
    json.dump(export_payload, fh, indent=2)

print(f"Export saved to: {export_path}")
with open(export_path) as fh:
    first_64 = fh.read(64)
print(f"First 64 bytes  : {first_64!r}")

In [ ]:
# Step 3 — Import into the target catalog
#
# ConfigImportRequest wraps the export payload. The platform applies each config
# individually; immutable fields that are already set in the target are skipped
# (reported in the errors list as skip/immutable, not as failures).
import_request = {
    "source": export_payload,
    # Optional: "overwrite": True to force-overwrite existing configs
}

resp = client.post(
    f"/admin/configs/{TARGET_CATALOG_ID}/import",
    json=import_request,
)
assert resp.status_code == 200, (
    f"Expected 200, got {resp.status_code}: {resp.text[:400]}"
)

import_result = resp.json()
applied = import_result.get("applied", [])
errors  = import_result.get("errors",  [])

print(f"Import to {TARGET_CATALOG_ID!r}:")
print(f"  applied : {applied}")
print(f"  errors  : {errors}")
print()
print(json.dumps(import_result, indent=2))

In [ ]:
# Step 4 — Verify routing resolution in the target catalog
resp = client.get(
    f"/configs/catalogs/{TARGET_CATALOG_ID}/collections/{COLLECTION_ID}/configs/routing/effective"
)
assert resp.status_code == 200, (
    f"Expected 200, got {resp.status_code}: {resp.text[:400]}"
)

effective = resp.json()
resolved  = effective.get("resolved", {})
sources   = effective.get("sources",  {})

print(f"Effective routing in {TARGET_CATALOG_ID!r}:")
print(json.dumps(resolved, indent=2))
print()

# Verify the source annotation indicates the value came from catalog or collection,
# not just from platform defaults (which would mean the import did nothing).
source_levels = set(sources.values())
assert source_levels, "No source annotations returned — check API version"
print(f"Source levels in target routing: {source_levels}")

if "platform" in source_levels and not source_levels - {"platform"}:
    print("  WARNING: all fields sourced from platform defaults — import may not have applied.")
else:
    print("  Import verified: at least one field is sourced above platform default.")

## Edge cases

### Case A — Immutable field already set in target

Some config fields are marked immutable once set (e.g. `partition_type` on a
PostgreSQL driver after the table is created). If the target already has such a
field set, the import should skip it and report it in the `errors` list with a
`skip_immutable` reason — not fail the entire import.

In [ ]:
# Verify immutable-field skip behavior
#
# If the target already has driver:postgresql with partition_type=LIST and the
# import tries to set it to RANGE, the platform should skip that field.
# We simulate this by importing the same payload a second time.

resp2 = client.post(
    f"/admin/configs/{TARGET_CATALOG_ID}/import",
    json=import_request,
)
assert resp2.status_code == 200, (
    f"Expected 200 on re-import, got {resp2.status_code}: {resp2.text[:400]}"
)

reimport_result = resp2.json()
reimport_errors = reimport_result.get("errors", [])

print("Re-import result (same payload, idempotency check):")
print(f"  applied : {reimport_result.get('applied', [])}")
print(f"  errors  : {reimport_errors}")
print()

# Immutable fields already set should appear in errors as skip, not as failures
skip_errors = [
    e for e in reimport_errors
    if isinstance(e, dict) and e.get("reason") in ("skip_immutable", "already_set", "no_change")
]
other_errors = [e for e in reimport_errors if e not in skip_errors]

if skip_errors:
    print(f"  Immutable skip entries: {skip_errors}")
if other_errors:
    print(f"  Non-skip errors (unexpected): {other_errors}")
if not reimport_errors:
    print("  No errors on re-import — platform is fully idempotent for this payload.")

In [ ]:
# export_timestamp is informational only — no version locking
#
# The export_timestamp records when the snapshot was taken. It does NOT prevent
# a later import of the same payload (no ETag or If-Match enforcement).
# This means the same export.json can be re-imported safely after the source
# catalog has been updated — the import will apply only the delta.

# Mutate the timestamp and re-import — must still return 200
tampered_payload = dict(export_payload)
tampered_payload["export_timestamp"] = "1970-01-01T00:00:00Z"  # obviously stale

resp3 = client.post(
    f"/admin/configs/{TARGET_CATALOG_ID}/import",
    json={"source": tampered_payload},
)
assert resp3.status_code == 200, (
    f"Expected 200 even with old timestamp, got {resp3.status_code}: {resp3.text[:400]}"
)

print(f"Re-import with stale timestamp → {resp3.status_code} (no version lock applied)")
print("export_timestamp is informational only — confirmed.")

client.close()